# Figure C.1 — Radon inversion, faststack and the triangulation map on two scan laws

**In the paper:** Fig. C.1; App. C.2 (Eqs. C.1–C.3); Sect. 3.3 (Eq. 13); the Pleiades law of Sect. 4.4.

The windows of a static scene are, up to the LSF, projections of the sky: its Radon transform.
Tomography inverts them by ramp-filtered backprojection. This notebook does that inversion, faststack
and the paper's map on the same windows, through a well-sampled scan law and through the Pleiades law.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the two scan laws, the angular weights δψ_j, and each window's zero point | `scanlaw.load_scan_law`, `radon.direction_weights`, `sim.random_zero_points` | App. C.2 |
| 2 | the windows: the same scene, the same seed | `sim.simulate_window`; in one call `sim.harvest_campaign` | — |
| 3 | the sinogram and the projection–slice theorem | `ins.render_curve`, `ins.project_al` | Eqs. C.1, C.2 |
| 4 | the inversion: the ramp filter, the weights, the backprojection | `radon.ramp_filter`; in one call `radon.filtered_backprojection` | Eq. C.3 |
| 5 | the forward route on the same windows | `dyn.clean_detection_set`, `joint.fit_joint` | Figs. 2, 3 |
| 6 | what the images show: the companion over the spurious level | — | App. C.2 |
| 7 | faststack | `radon.faststack` | App. C.2; Harrison 2011 |
| 8 | the paper's map of the detections, and the figure | `bp.backprojection_map` | Fig. C.1 |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**The scene.** Fig. 2's pair (200 mas, PA 110°, 𝒮 = 0.2) on the toy instrument at S/N 100, static,
every window at its own sampling phase. The laws: the GOST forecast at ecliptic latitude +60°
(95 transits, no gap above 10°) and at the Pleiades (47 transits, a 90° gap).

**What it reproduces** (the caption of Fig. C.1, App. C.2). 95 and 47 transits; the companion over the
spurious level 8.1 (inversion) and 21.5 (faststack) on the +60° law, 1.1 and 14.2 on the Pleiades law;
the map's peak within 5 mas of the companion on both. A few seconds.

In [ ]:
# the imports: numpy, matplotlib, and the scanlib modules; every scanlib call below carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import backprojection as bp   # the triangulation map and its peak
from scanlib import radon                  # the inverse Radon transform and faststack
from scanlib import dynamics as dyn        # the fits to the detections (the sinogram fits)
from scanlib import joint                  # the joint fit of all windows, and its helpers

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in auxil/sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

SAVE_PDF = False                           # True: write pdf/figC1_inverse_radon.pdf

In [ ]:
# the scene (Figure 2's pair) and its noise, the two scan laws, and the image grid
RHO_MAS = 200.0                            # Figure 2's companion: its separation from the primary [mas]
PA_DEG = 110.0                             # its position angle [deg, from north through east]
Q = 0.2                                    # its flux ratio to the primary
SCENE = sim.Scene.pair(RHO_MAS, PA_DEG, Q)
# SCENE.positions holds one row per source, the primary first; [1] is the second row, the companion's
TRUTH = SCENE.positions[1]                 # the companion's true sky offset (dalpha*, ddelta) [mas]
SEED = 2                                   # Figure 2's noise seed
# the seed of the zero points' stream, a list of two whole numbers: [the noise seed, the label 3 of this stream]
ZERO_POINT_SEED = [SEED, 3]                # the windows' zero points: one draw per law
LADDER_K_MAX = 2                           # the forward route: a pair
SIGMA_NOISE = sim.PRIMARY_FLUX / 100.0         # the toy regime, S/N 100: the noise sigma per sample
# LAWS: a table with one row per scan law: (key, scan-law file, label), the well-sampled case first
LAWS = (("well-sampled", scanlaw.BETA60, "ecliptic latitude +60 deg"),
        ("pathological", scanlaw.PLEIADES, "Pleiades centre"))
# the image square, kept inside the windows' reach (a pixel at most IMAGE_HALF_MAS sqrt(2)
# from the origin, under a window's half-length)
IMAGE_HALF_MAS = 350.0                     # the image's half-width [mas]
IMAGE_STEP_MAS = 2.0                       # the image's pixel [mas]

## 1. The two scan laws and the angular weights (App. C.2)

A scan at ψ + 180° measures the projection at ψ with η reversed, so the directions are folded onto the
half-turn [0, π). Each transit's weight δψ_j is half the gap to each neighbour on the folded circle,
wrapping at 0 and π; the weights sum to π, and are π/N for N evenly spaced directions.

In [ ]:
# each law's scan directions, the largest gap on the half-turn, and the angular weights delta psi_j of Eq. C.3
# delta psi_j for N evenly spaced directions is pi/N (App. C.2)
assert np.allclose(radon.direction_weights(np.arange(36) * np.pi / 36), np.pi / 36)
# laws: a table from a law's key ("well-sampled", "pathological") to that law's own table. This cell gives each
# law four entries: "psi" (scan angles [rad]), "label", "weights" (delta psi_j [rad]) and "eta0" (zero points
# [mas]); the cells below add the windows, the images and the numbers under further names
laws = {}
for key, path, label in LAWS:
    # scanlaw.load_scan_law: scan angles [rad] and epochs (not used: the scene is static)
    psi, _epochs = scanlaw.load_scan_law(path)
    folded = np.sort(np.mod(psi, np.pi))   # scan directions reduced to [0, pi), sorted
    # gaps between neighbouring directions, including the wrap-around: np.r_ adds the first direction plus pi
    # after the last
    gaps = np.diff(np.r_[folded, folded[0] + np.pi])
    # radon.direction_weights: delta psi_j, each transit's share of the half-turn
    weights = radon.direction_weights(psi)
    # sim.random_zero_points: each window's sampling phase eta0_j. rng_zero_points is made anew for each law from
    # the same seed, ZERO_POINT_SEED
    rng_zero_points = np.random.default_rng(ZERO_POINT_SEED)
    eta0 = sim.random_zero_points(rng_zero_points, psi.size, inst)
    laws[key] = dict(psi=psi, label=label, weights=weights, eta0=eta0)
    print(f"{key:13s} ({label}): {psi.size} transits; largest gap on the half-turn "
          f"{np.rad2deg(gaps.max()):.1f} deg; weights sum to {weights.sum():.6f} (pi = {np.pi:.6f})")

## 2. The windows

The scene at every transit of each law, S/N 100, Fig. 2's seed, each window at its own zero point η₀ⱼ
(a source at η is seen at η − η₀ⱼ). `sim.harvest_campaign` renders the same windows (and reduces them, §5).

In [ ]:
# the windows of each law, rendered by hand and by sim.harvest_campaign from the same seed; the campaign, which also
# holds the detections of the per-window reduction, is kept for section 5
# law: one law's table (its entries are listed where laws is made)
for key, law in laws.items():
    # render by hand, one generator per law: rng_noise, seeded by SEED alone (the same seed for both laws); the
    # windows draw from it in the order of the transits
    rng_noise = np.random.default_rng(SEED)
    windows = []
    for psi_j, zero_point in zip(law["psi"], law["eta0"]):
        sigma_samples = SIGMA_NOISE        # one number: the same noise sigma for every sample
        # sim.simulate_window: the noise-free window of the scene at these positions plus one noise draw; the
        # positions are in the window's frame, eta - eta0_j
        samples = sim.simulate_window(SCENE.along_scan(psi_j) - zero_point, SCENE.fluxes, inst, rng_noise,
                                      sigma_samples)
        windows.append(samples)
    law["windows"] = windows
    law["sigma_noise"] = sigma_samples
    # sim.harvest_campaign: the same windows in one call, and their per-window reduction (the detections); a new
    # generator from the same seed, so that it draws the same noise
    rng_noise_one_call = np.random.default_rng(SEED)
    campaign = sim.harvest_campaign(SCENE, law["psi"], rng_noise_one_call, inst, SIGMA_NOISE, k_max=LADDER_K_MAX,
                                    return_windows=True, eta0=law["eta0"])
    # same windows, bit for bit (compared window by window)
    assert all(np.array_equal(a, b) for a, b in zip(campaign.windows, windows))
    law["campaign"] = campaign
    print(f"{key:13s}: {len(windows)} windows x {windows[0].size} samples, noise sigma {sigma_samples:.3f}")
print("sim.harvest_campaign renders the same windows")

## 3. The sinogram (Eq. C.1) and the projection–slice theorem (Eq. C.2)

For point sources the expected profile at scan angle ψ is

$$s(\eta\mid\psi) = \sum_i f_i\,\mathcal P\!\left[\eta - \rho_i\cos(\psi-\varphi_i)\right], \qquad ({\rm C.1})$$

each source a sinusoid in (ψ, η). Its Fourier transform along η is one line through the Fourier
transform of the blurred sky image $I$:

$$\hat s(\nu\mid\psi) = \hat I(\nu\sin\psi,\ \nu\cos\psi). \qquad ({\rm C.2})$$

The cell checks C.2 at one direction and three frequencies. The figure's sinogram is the noise-free
C.1 over the half-turn; the directions each law samples are ticks on its edges.

In [ ]:
# Eq. C.2 at psi = 70 deg: the profile per mas (Eq. C.1) Fourier-transformed along eta, against the 2-D transform
# of the blurred scene, sum_i f_i exp(-2 pi i k.x_i) exp(-2 pi^2 sigma^2 |k|^2), on the line k = nu (sin psi, cos psi)
SIGMA_PSF = FWHM_MAS / (2.0 * np.sqrt(2.0 * np.log(2.0)))      # the Gaussian PSF's sigma [mas]
PITCH = float(np.diff(inst.sample_grid_mas)[0])                 # the sample pitch [mas]: the first step of the grid
psi_70 = np.deg2rad(70.0)                                       # the direction of the check [rad]
eta_wide = np.arange(-1500.0, 1500.0 + 1e-9, 1.0)               # wide enough to hold every tail; 1 mas steps
# ins.render_curve: the noise-free model profile at any eta [flux per sample]; over the pitch, a density per mas
density = ins.render_curve(eta_wide, SCENE.along_scan(psi_70), SCENE.fluxes, inst) / PITCH
# nu: the spatial frequency along the scan
for nu in (0.002, 0.005, 0.01):                                                # cycles per mas
    # s_hat: the left side of Eq. C.2, the profile's Fourier transform at nu
    s_hat = np.sum(density * np.exp(-2j * np.pi * nu * eta_wide))              # the 1 mas step is the integral's d eta
    # k: the point of the 2-D frequency plane on the line of this direction, nu (sin psi, cos psi)
    k = nu * np.array([np.sin(psi_70), np.cos(psi_70)])
    # I_hat: the right side, the transform of the point sources times the transform of the Gaussian PSF
    sources_hat = np.sum(SCENE.fluxes * np.exp(-2j * np.pi * SCENE.positions @ k))
    psf_hat = np.exp(-2.0 * np.pi ** 2 * SIGMA_PSF ** 2 * nu ** 2)
    I_hat = sources_hat * psf_hat
    assert abs(s_hat - I_hat) < 1e-9 * SCENE.fluxes.sum()
print("Eq. C.2 holds at psi = 70 deg, nu = 0.002, 0.005, 0.01 per mas")

In [ ]:
# the figure's sinogram: the noise-free model of Eq. C.1 over the half-turn, and each law's sampled directions
# folded onto it
PSI_FINE = np.linspace(0.0, np.pi, 361)              # the directions of the sinogram [rad]: every half degree
ETA_FINE = np.arange(-350.0, 350.0 + 1e-9, 2.0)      # the images' half-width
# one noise-free profile (Eq. C.1) per direction of PSI_FINE
profiles = []
for psi_fine in PSI_FINE:
    profiles.append(ins.render_curve(ETA_FINE, SCENE.along_scan(psi_fine), SCENE.fluxes, inst))
# .T turns the table: one row per eta, one column per direction
SINO_MODEL = np.array(profiles).T                    # (eta, psi)
for key, law in laws.items():
    # the sampled directions folded onto [0, pi) [deg], one tick per transit in the figure
    law["folded_deg"] = np.rad2deg(np.mod(law["psi"], np.pi))
# psi = 180 deg: the 0 deg profile reversed. SINO_MODEL[:, 0]: the first column, the profile at 0 deg;
# SINO_MODEL[::-1, -1]: the last column, the profile at 180 deg, read from its end
assert SINO_MODEL.shape == (ETA_FINE.size, PSI_FINE.size) and np.isclose(SINO_MODEL[:, 0], SINO_MODEL[::-1, -1]).all()
print(f"model sinogram: {PSI_FINE.size} directions over the half-turn x {ETA_FINE.size} positions; the 180 deg profile is the 0 deg one reversed")

In [ ]:
# the companion's along-scan offset over each law's transits, and how often it lies within one FWHM of the primary
for key, law in laws.items():
    # ins.project_al: eta = dra* sin psi + ddec cos psi (Eq. 9); TRUTH[0] is the companion's dra*, TRUTH[1] its ddec
    track = ins.project_al(TRUTH[0], TRUTH[1], law["psi"])
    print(f"{key:13s}: companion's projected offset ranges over "
          f"{track.min():+.0f} to {track.max():+.0f} mas; "
          f"{(np.abs(track) < FWHM_MAS).sum()} of {track.size} transits within one FWHM of the primary")

## 4. The inversion (Eq. C.3)

$$I(\Delta\alpha^*, \Delta\delta) \approx \sum_j q\!\left(\Delta\alpha^*\sin\psi_j + \Delta\delta\cos\psi_j \mid \psi_j\right)\delta\psi_j,
\qquad \hat q(\nu\mid\psi) \equiv |\nu|\,\hat s(\nu\mid\psi). \qquad ({\rm C.3})$$

(1) The ramp filter: each window as a density (flux per mas), multiplied by |ν| in Fourier space and
returned on a grid eight times finer than the samples (`radon.ramp_filter`); without it, backprojection
returns each source as a 1/r halo. (2) The weights δψ_j of §1. (3) The backprojection: each pixel takes,
from each window, the filtered value at its own along-scan coordinate, $\Delta\alpha^*\sin\psi_j +
\Delta\delta\cos\psi_j - \eta_{0j}$. Checked against `radon.filtered_backprojection`, and against the folded form
of App. C.2 (directions on [0, π), the other half's windows reversed), equal wherever every window
reaches.

In [ ]:
# the inversion of Eq. C.3 written out, in the direct and the folded form, and checked against the package's
# radon.filtered_backprojection
axis = np.arange(-IMAGE_HALF_MAS, IMAGE_HALF_MAS + 1e-9, IMAGE_STEP_MAS)   # the pixel centres along each axis [mas]
# the sky offset of every pixel [mas]: pixel_dra changes along the columns, pixel_ddec along the rows
pixel_dra, pixel_ddec = np.meshgrid(axis, axis)
for key, law in laws.items():
    # the direct sum (image) and the folded form (image_folded)
    image = np.zeros_like(pixel_dra)
    image_folded = np.zeros_like(pixel_dra)
    for psi_j, weight_j, samples, zero_point in zip(law["psi"], law["weights"], law["windows"], law["eta0"]):
        # radon.ramp_filter: the window's ramp-filtered projection on a grid 8x finer: the grid's positions
        # eta_fine [mas, in the window's frame] and the filtered values q (the q of Eq. C.3)
        eta_fine, q = radon.ramp_filter(samples, inst.sample_grid_mas)  # 1.
        # along-scan coordinate of every pixel in this window's own frame
        pixel_eta = pixel_dra * np.sin(psi_j) + pixel_ddec * np.cos(psi_j) - zero_point   # 3., weighted by 2.
        # each pixel collects the filtered projection at its coordinate, weighted; a pixel outside the window
        # collects zero (left=0.0, right=0.0)
        image += weight_j * np.interp(pixel_eta, eta_fine, q, left=0.0, right=0.0)
        # the folded form: the direction on [0, pi); the other half's window reversed, its zero point -zero_point
        other_half = np.mod(psi_j, 2 * np.pi) >= np.pi       # True: this scan's direction lies in [pi, 2 pi)
        if other_half:
            samples_folded = samples[::-1]                   # the window read from its end
            zero_point_folded = -zero_point
        else:
            samples_folded = samples
            zero_point_folded = zero_point
        eta_fine_folded, q_folded = radon.ramp_filter(samples_folded, inst.sample_grid_mas)
        psi_folded = np.mod(psi_j, np.pi)
        pixel_eta_folded = pixel_dra * np.sin(psi_folded) + pixel_ddec * np.cos(psi_folded) - zero_point_folded
        image_folded += weight_j * np.interp(pixel_eta_folded, eta_fine_folded, q_folded,
                                             left=0.0, right=0.0)
    # radon.filtered_backprojection: the same image in one call, with its axis
    package_image, package_axis = radon.filtered_backprojection(law["psi"], law["windows"], inst, half=IMAGE_HALF_MAS,
                                                                step=IMAGE_STEP_MAS, eta0=law["eta0"])
    # the hand-built sum equals the package's exactly
    assert np.array_equal(package_axis, axis) and np.array_equal(package_image, image)
    # the folded form agrees to machine precision, relative to the peak, within every
    # window's reach (its half-length less the largest zero point)
    # inst.sample_grid_mas[-1]: the last sample's position, the window's half-length [mas]
    reach = np.hypot(pixel_dra, pixel_ddec) < (inst.sample_grid_mas[-1] - np.abs(law["eta0"]).max())
    folded_difference = np.abs(image_folded - image)[reach].max() / image.max()
    assert folded_difference < 1e-12
    law["image"] = image
    print(f"{key:13s}: radon.filtered_backprojection reproduces the sum exactly; "
          f"the folded form differs by {folded_difference:.1e} of the peak")

In [ ]:
# one filtered window, to show the grid it lands on
# the first window ([0]) of the Pleiades law: the fine grid's positions [mas] and the filtered values q
eta_fine, q = radon.ramp_filter(laws["pathological"]["windows"][0], inst.sample_grid_mas)
print(f"one filtered window: {q.size} points from {eta_fine[0]:.1f} to {eta_fine[-1]:.1f} mas")

## 5. The forward route on the same windows (Figs. 2, 3)

The paper's route on these windows: per-window reduction, the detections cleaned on the map, the joint
window fit of the pair. Its position is marked on the images. `dyn.clean_detection_set` is the cleaning
loop of Fig. 2 §5 in one call: fit Eq. 11 to the kept detections, take the map's peak, keep the lines
through it, repeat until the kept set stops changing. For a static source nothing moves the detections
between passes (its moving form applies Eq. 17 there), so the map is the same at every pass: one map,
one prune, one fit.

In [ ]:
# the paper's forward route on each law: clean the detections on the map (static), then fit the pair's windows
# jointly, seeded there
for key, law in laws.items():
    campaign = law["campaign"]
    # dyn.clean_detection_set returns four things: the seed (Eq. 11 on the kept detections), its covariance (not
    # used here), the mask of the kept detections, and a log (not used here). The second argument, the
    # detections' epochs, is None: the source is static
    seed, _seed_covariance, keep, _log = dyn.clean_detection_set(campaign.psi, None, campaign.eta, campaign.sigma,
                                                                 static=True)
    # joint.fit_joint: the window fit, Eq. 18
    fit = joint.fit_joint(campaign.windows, law["psi"], seed, inst, campaign.sigma_noise, eta0=campaign.eta0)
    law["fit"] = fit
    # truth: the fitted offset and its 1-sigma errors [mas], printed beside the companion's true offset
    fitted_offset = fit["theta"]           # the companion's fitted (dalpha*, ddelta) [mas]
    fitted_sigma = fit["sigma_theta"]      # their errors [mas]
    # fit["flux"][1] / fit["flux"][0]: the companion's fitted flux over the primary's
    print(f"{key:13s}: {campaign.eta.size} detections, {int(keep.sum())} kept; fit ({fitted_offset[0]:.1f} +/- "
          f"{fitted_sigma[0]:.1f}, {fitted_offset[1]:.1f} +/- {fitted_sigma[1]:.1f}) mas; "
          f"truth ({TRUTH[0]:.1f}, {TRUTH[1]:.1f}); "
          f"q {fit['flux'][1] / fit['flux'][0]:.3f}")

## 6. What the images show (App. C.2)

The image's value at the true companion over its **spurious level** (`grain` in the code), the largest
excursion, positive or negative, farther than 1.5 FWHM from both sources (the streaks and the ramp's
troughs). The same ratio for faststack follows in §7.

In [ ]:
# truth: the inverse Radon image's contrast on each law: its value at the true companion over the spurious level
for key, law in laws.items():
    image = law["image"]
    # image value at the pixel nearest the true companion: the row index is from ddelta (TRUTH[1]), the column
    # index from dalpha* (TRUTH[0])
    companion_row = np.argmin(np.abs(axis - TRUTH[1]))
    companion_column = np.argmin(np.abs(axis - TRUTH[0]))
    at_companion = image[companion_row, companion_column]
    # mask of pixels more than 1.5 FWHM from both sources: from the primary (the origin) and from the companion
    far_from_primary = np.hypot(pixel_dra, pixel_ddec) > 1.5 * FWHM_MAS
    far_from_companion = np.hypot(pixel_dra - TRUTH[0], pixel_ddec - TRUTH[1]) > 1.5 * FWHM_MAS
    far = far_from_primary & far_from_companion
    # grain: the largest excursion, positive or negative, in that region
    grain = np.abs(image[far]).max()
    law["contrast"] = at_companion / grain
    print(f"{key:13s}: image at the companion {at_companion:.3f}, at the primary {image.max():.3f} "
          f"(flux/mas^2); grain {grain:.3f}; companion / grain = {at_companion / grain:.1f}")

## 7. Faststack (App. C.2; Harrison 2011)

A robust stack of the window samples: each pixel takes the sample of every window whose strip contains it,
and the low values weight down the samples a bright neighbour fell into (written out in
`figC2_three_pictures.ipynb` §5). No filter, no inversion. The same two numbers as §6, and faststack's value
halfway between the stars. The small plot is a first look; the figure draws faststack as its third column.

In [ ]:
# Faststack of the same windows on both laws: its contrast against the inverse Radon image's,
# and its plateau between the stars; the image is kept for the figure's third column.
# the pixel halfway between the primary (the origin) and the true companion: row from ddelta, column from dalpha*
mid_row = np.argmin(np.abs(axis - TRUTH[1] / 2))
mid_column = np.argmin(np.abs(axis - TRUTH[0] / 2))
# PAPER_STYLE: a table of matplotlib settings, from a setting's name to its value
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    fig_fs, ax_fs = plt.subplots(1, 2, figsize=(7.0, 3.4))
    # one panel per law
    for ax, (key, law) in zip(ax_fs, laws.items()):
        # radon.faststack: the robust stack of the window samples; [0] is the image (then its variance, counts, axis)
        faststack_image = radon.faststack(law["psi"], law["windows"], inst, half=IMAGE_HALF_MAS, step=IMAGE_STEP_MAS,
                                          eta0=law["eta0"])[0]
        law["faststack"] = faststack_image
        # truth: the image at the true companion (row from ddelta, column from dalpha*) over the spurious level;
        # far is the mask of section 6, the pixels more than 1.5 FWHM from both sources (the same grid)
        at_companion = faststack_image[np.argmin(np.abs(axis - TRUTH[1])), np.argmin(np.abs(axis - TRUTH[0]))]
        law["fs_contrast"] = at_companion / np.abs(faststack_image[far]).max()
        print(f"{key:13s}: companion / grain -- faststack {law['fs_contrast']:5.1f}, "
              f"inverse Radon {law['contrast']:5.1f}; faststack halfway between the stars "
              f"{faststack_image[mid_row, mid_column] / at_companion:.2f} of its value at the companion")
        # the image, sqrt stretch, negative values clipped to zero; East to the left
        ax.imshow(np.sqrt(np.clip(faststack_image, 0, None)), origin="lower", cmap="gray_r",
                  extent=(axis[0], axis[-1], axis[0], axis[-1]), interpolation="nearest")
        ax.set_xlim(axis[-1], axis[0])
        ax.set_ylim(axis[0], axis[-1])
        # the true companion: *TRUTH passes its two numbers (dalpha*, ddelta) as x and y
        ax.scatter(*TRUTH, marker="*", s=120, facecolor="white", edgecolor="k", lw=0.8, zorder=6)
        ax.set_xlabel(r"$\Delta\alpha^{*}$ [mas]")
        ax.set_title(f"faststack, {key}", fontsize=9)
    ax_fs[0].set_ylabel(r"$\Delta\delta$ [mas]")
    plt.tight_layout()
    plt.show()

## 8. The figure (Fig. C.1)

Rows: the +60° law, the Pleiades law. Columns: the sinogram with the sampled directions as edge ticks;
the inversion; faststack; the paper's map of the reduced detections. Plotting only.

In [ ]:
# the figure's style: the fonts, the markers, the sinogram's extent and the length of its edge ticks
plt.rcParams.update({"font.size": 10.0, "axes.linewidth": 0.5,
                     "xtick.labelsize": 9.0, "ytick.labelsize": 9.0})
# marker styles: truth (star), primary (plus), forward fit (cross)
TRUTH_MK = dict(marker="*", s=120, facecolor="white", edgecolor="k", lw=0.8, zorder=6)
PRIMARY_MK = dict(marker="+", s=80, color="white", lw=1.2, zorder=6)   # white: it sits on its own image
FIT_MK = dict(marker="x", s=70, color="k", lw=1.5, zorder=7)            # on top of the star: the fit is ~1 mas off
# the model sinogram's extent (psi deg, eta mas) and the edge ticks' length
SINO_EXTENT = (0.0, 180.0, ETA_FINE[0], ETA_FINE[-1])
TICK_MAS = 30.0                                       # length of a direction tick on the top and bottom edges
SINO_PEAK = SINO_MODEL.max()

In [ ]:
# the paper's own map of the same windows: bp.backprojection_map of the reduced detections (Fig. 2's object); the
# primary is fitted and removed per window, so the map holds the companion's ridges only
for key, law in laws.items():
    campaign = law["campaign"]
    # bp.backprojection_map: the map of Eq. 13 of the detections, with its axis and extent; 350.0 and 2.0: the
    # map's half-width and step [mas]
    law["map"], law["map_axis"], law["map_extent"] = bp.backprojection_map(campaign.psi, campaign.eta, campaign.sigma,
                                                                           350.0, 2.0)
    # peak_row_column: the (row, column) of the map's largest pixel; the row is ddelta, the column dalpha*
    peak_row_column = np.unravel_index(np.argmax(law["map"]), law["map"].shape)
    peak_dalpha = law["map_axis"][peak_row_column[1]]
    peak_ddelta = law["map_axis"][peak_row_column[0]]
    law["map_peak"] = np.array([peak_dalpha, peak_ddelta])        # (dalpha*, ddelta) of the map's peak
    law["n_det"] = int(campaign.psi.size)
    print(f"{key:13s}: {law['n_det']} detections of {law['psi'].size} windows; the map's peak at "
          f"({law['map_peak'][0]:.0f}, {law['map_peak'][1]:.0f}) mas, truth ({TRUTH[0]:.0f}, {TRUTH[1]:.0f})")
# for each law, the distance of the map's peak from the true companion [mas]
assert all(np.hypot(*(law["map_peak"] - TRUTH)) < 5.0 for law in laws.values()), "the map's peak is not at the companion"
print("the map's peak lies within 5 mas of the companion for both laws")

In [ ]:
# Two rows, one per scan law: the model sinogram over the half-turn with the law's sampled directions as
# edge ticks; the filtered back-projection of the measured windows with the truth and the forward fit marked;
# faststack of the same windows; the paper's own map of the same windows, the truth marked.
# --- FAST cell: plotting only. ---
# row per law: sinogram, inverse Radon image, faststack, the paper's map; A&A text width
IMG_KW = dict(origin="lower", cmap="gray_r", interpolation="nearest", rasterized=True)
fig, axes = plt.subplots(2, 4, figsize=(17.0 / 2.54, 3.3), constrained_layout=True)
for row, (key, law) in enumerate(laws.items()):
    ax_s, ax_i, ax_f, ax_m = axes[row]
    # the model sinogram over the half-turn, sqrt stretch, the same image in both rows
    ax_s.imshow(np.sqrt(SINO_MODEL / SINO_PEAK), extent=SINO_EXTENT, aspect="auto", vmin=0, vmax=1, **IMG_KW)
    ax_s.vlines(law["folded_deg"], ETA_FINE[-1] - TICK_MAS, ETA_FINE[-1], color="k", lw=0.6)
    ax_s.vlines(law["folded_deg"], ETA_FINE[0], ETA_FINE[0] + TICK_MAS, color="k", lw=0.6)
    ax_s.set_xlim(0, 180)
    ax_s.set_xticks([0, 45, 90, 135, 180])
    ax_s.set_ylim(ETA_FINE[0], ETA_FINE[-1])
    ax_s.set_ylabel(r"$\eta$ [mas]")
    # the filtered back-projection, linear from zero (white) to its peak (black); negative lobes clipped
    image = law["image"]
    ax_i.imshow(image, extent=(axis[0], axis[-1], axis[0], axis[-1]), vmin=0.0, vmax=image.max(), **IMG_KW)
    # faststack, linear from zero (white) to its own peak; the noise floor below zero clipped
    faststack_image = law["faststack"]
    ax_f.imshow(faststack_image, extent=(axis[0], axis[-1], axis[0], axis[-1]), vmin=0.0,
                vmax=np.nanmax(faststack_image), **IMG_KW)
    # the paper's map, linear from zero to its peak
    ax_m.imshow(law["map"], extent=law["map_extent"], vmin=0.0, vmax=law["map"].max(), **IMG_KW)
    for ax_ in (ax_i, ax_f, ax_m):
        ax_.set_xlim(axis[-1], axis[0])                                           # East to the left
        ax_.set_ylim(axis[0], axis[-1])
        ax_.set_xticks([-200, 0, 200])
        ax_.set_yticks([-200, 0, 200])
        ax_.scatter(*TRUTH, **TRUTH_MK)
    for ax_ in (ax_i, ax_f):
        ax_.scatter(0, 0, **PRIMARY_MK)                                             # white on its own image
    ax_m.scatter(0, 0, **dict(PRIMARY_MK, color="k"))                               # black: the map is white there (the primary is removed)
    ax_i.set_ylabel(r"$\Delta\delta$ [mas]")
    for ax_ in (ax_f, ax_m):
        ax_.tick_params(labelleft=False)
    for ax_ in (ax_s, ax_i, ax_f, ax_m):
        ax_.tick_params(direction="in", top=True, right=True)
        ax_.set_box_aspect(1)                                                     # eight squares
    # the panel letters: ("abcd", "efgh")[row] is this row's four letters; letter_y is each letter's height in
    # its panel, the sinogram's letter below its tick row
    for ax_, letter, letter_y in zip((ax_s, ax_i, ax_f, ax_m), ("abcd", "efgh")[row], (0.90, 0.96, 0.96, 0.96)):
        ax_.text(0.04, letter_y, f"({letter})", transform=ax_.transAxes, va="top", ha="left", fontsize=9,
                 fontweight="bold",
                 bbox=dict(boxstyle="round", fc="white", ec="none", alpha=0.85))
# x labels on the bottom row only; the three sky panels share one, under the middle panel
axes[1, 0].set_xlabel(r"scan direction $\psi$ [deg]")
axes[1, 2].set_xlabel(r"$\Delta\alpha^{*}$ [mas]  (East $\leftarrow$)")
for ax_ in axes[0]:
    ax_.tick_params(labelbottom=False)

In [ ]:
# the numbers for the caption, printed for each law and asserted
for key, law in laws.items():
    fitted_offset = law["fit"]["theta"]            # the companion's fitted (dalpha*, ddelta) [mas]
    fitted_sigma = law["fit"]["sigma_theta"]       # their errors [mas]
    # numbers for the caption
    print(f"{key} ({law['label']}): {law['psi'].size} transits, {law['n_det']} detections; companion / grain -- "
          f"inverse Radon {law['contrast']:.1f}, faststack {law['fs_contrast']:.1f}; the map's peak "
          f"({law['map_peak'][0]:.0f}, {law['map_peak'][1]:.0f}) mas; "
          f"fit ({fitted_offset[0]:.1f} +/- {fitted_sigma[0]:.1f}, {fitted_offset[1]:.1f} +/- {fitted_sigma[1]:.1f}) mas")

# the caption's numbers
well_sampled = laws["well-sampled"]        # the +60 deg law's table
pathological = laws["pathological"]        # the Pleiades law's table
assert (well_sampled["psi"].size, pathological["psi"].size) == (95, 47)
assert (round(well_sampled["contrast"], 1), round(well_sampled["fs_contrast"], 1)) == (8.1, 21.5)
assert (round(pathological["contrast"], 1), round(pathological["fs_contrast"], 1)) == (1.1, 14.2)
print("checks passed")

In [ ]:
# write the PDF only when SAVE_PDF is True
if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "figC1_inverse_radon.pdf", bbox_inches="tight")
    print("wrote pdf/figC1_inverse_radon.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")